# Assignment 4 - Part III: Prognosis

Prognosis is the third PHM step: predict the Remaining Useful Life (RUL) of an asset from sensor history. The dataset is NASA's C-MAPSS FD001 turbofan benchmark. Each engine unit runs from a healthy state to failure, generating per-cycle sensor and operating-condition readings.

We compare a classical regressor (SVR) against a deep one (LSTM) on identical preprocessed windows.

## What this notebook covers for the different grades

- **Grade 3:** download (already local) and describe the FD001 dataset - units, cycles, sensors, RUL distribution.
- **Grade 4:** train an RBF-kernel SVR on flattened sensor windows to predict RUL, score it with RMSE / MAE / R² / C-MAPSS score.
- **Grade 5:** train a stacked LSTM as the DL method on the same windowed data, score with the same metrics, and present a side-by-side comparison.

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.svm import SVR
import joblib
from tqdm.auto import tqdm

from config import *
from helpers import build_cmapss_tensors, cmapss_metrics, load_cmapss

In [ ]:
print(f"Python    : {sys.version.split()[0]}")
print(f"PyTorch   : {torch.__version__}")
print(f"CUDA      : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device    : {torch.cuda.get_device_name(0)}")
print(f"Conda env : {os.environ.get('CONDA_DEFAULT_ENV', 'unknown')}")

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"dataset       : C-MAPSS {CMAPSS_DATASET}")
print(f"window size   : {CMAPSS_WINDOW} cycles")
print(f"RUL cap       : {CMAPSS_RUL_CAP}")
print(f"SVR           : C={SVR_C}, gamma={SVR_GAMMA}, epsilon={SVR_EPSILON}")
print(f"LSTM          : hidden={LSTM_HIDDEN_DIM}, layers={LSTM_NUM_LAYERS}, dropout={LSTM_DROPOUT}, epochs={LSTM_EPOCHS}")

## Grade 3 - describe the dataset

FD001 contains 100 engine units in both train and test. Each train unit runs from healthy to failure; each test unit is truncated at some earlier cycle, and the supplied `RUL_FD001.txt` gives the true RUL at that truncation point.

In [ ]:
train_df, test_df, rul_test = load_cmapss(CMAPSS_DIR, CMAPSS_DATASET)

print(f"train rows: {len(train_df)},  units: {train_df['unit_id'].nunique()}")
print(f"test rows : {len(test_df)},  units: {test_df['unit_id'].nunique()}")
print(f"RUL test  : {len(rul_test)} values, mean={rul_test.mean():.1f}, min={rul_test.min()}, max={rul_test.max()}")

cycles_per_unit = train_df.groupby('unit_id')['cycle'].max()
print(f"train life cycles: mean={cycles_per_unit.mean():.0f}, min={cycles_per_unit.min()}, max={cycles_per_unit.max()}")

In [ ]:
# Run the full preprocessing pipeline: load -> compute RUL -> drop constants ->
# -> scale on train only -> slide windows.
X_train, y_train, X_test, y_test, n_features = build_cmapss_tensors(
    data_dir=CMAPSS_DIR,
    dataset=CMAPSS_DATASET,
    window_size=CMAPSS_WINDOW,
    rul_cap=CMAPSS_RUL_CAP,
)
print(f"n_features kept : {n_features} (constant sensors dropped)")
print(f"X_train: {tuple(X_train.shape)}  y_train: {tuple(y_train.shape)}")
print(f"X_test : {tuple(X_test.shape)}  y_test : {tuple(y_test.shape)}")

## Grade 4 - Support Vector Regression

SVR can't take 3-D input. Each `(window, features)` window is flattened to a `window * features`-dim vector, then fed to an RBF-kernel SVR.

In [ ]:
X_train_flat = X_train.reshape(X_train.shape[0], -1).numpy()
X_test_flat  = X_test.reshape(X_test.shape[0], -1).numpy()
y_train_np   = y_train.numpy()
y_test_np    = y_test.numpy()
print(f"X_train_flat: {X_train_flat.shape}  X_test_flat: {X_test_flat.shape}")

In [ ]:
# Fit SVR, save to models/, then reload before scoring so the eval cell is decoupled from this training cell.
svr = SVR(kernel="rbf", C=SVR_C, gamma=SVR_GAMMA, epsilon=SVR_EPSILON)
print("Fitting the SVR...")
svr.fit(X_train_flat, y_train_np)

svr_path = MODELS_DIR / SVR_CHECKPOINT_NAME
joblib.dump(svr, svr_path)
print(f"saved {svr_path}")

In [ ]:
svr_loaded = joblib.load(MODELS_DIR / SVR_CHECKPOINT_NAME)
y_pred_svr = svr_loaded.predict(X_test_flat)
svr_metrics = cmapss_metrics(y_test_np, y_pred_svr)
print("SVR test metrics:")
for k, v in svr_metrics.items():
    print(f"  {k:>14s}: {v:.4f}")

## Grade 5 - LSTM regressor and head-to-head comparison

The DL method is a stacked LSTM that consumes each window as a length-30 sequence and regresses the RUL at the window's last cycle. Same train/test split as the SVR - the windows are literally the same tensors - so the comparison is apples to apples.

In [ ]:
class LSTMRulPredictor(nn.Module):
    def __init__(self, n_features, hidden=LSTM_HIDDEN_DIM, layers=LSTM_NUM_LAYERS,
                 dropout=LSTM_DROPOUT, head_hidden=LSTM_HEAD_HIDDEN):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden,
                            num_layers=layers, batch_first=True, dropout=dropout)
        self.head = nn.Sequential(
            nn.Linear(hidden, head_hidden), nn.ReLU(),
            nn.Linear(head_hidden, 1),
        )

    def forward(self, x):
        out, _ = self.lstm(x)               # (B, window, hidden)
        return self.head(out[:, -1, :]).squeeze(-1)  # (B,)

lstm_model = LSTMRulPredictor(n_features=n_features).to(device)
print(lstm_model)

In [ ]:
# Build train/val/test loaders directly from the windowed tensors.
full_train_ds = TensorDataset(X_train, y_train)

val_size = int(CMAPSS_VAL_RATIO * len(full_train_ds))
train_size = len(full_train_ds) - val_size

train_ds, val_ds = random_split(
    full_train_ds, [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED),
)
test_ds = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=LSTM_BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=LSTM_BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=LSTM_BATCH_SIZE)
print(f"train windows {len(train_ds)},  val {len(val_ds)},  test {len(test_ds)}")

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(lstm_model.parameters(), lr=LSTM_LR, weight_decay=LSTM_WEIGHT_DECAY)

train_hist, val_hist = [], []
best_val = float("inf")
patience = 0

for epoch in tqdm(range(LSTM_EPOCHS), desc="train LSTM", unit="epoch"):
    # Training phase
    lstm_model.train(); 
    total_train_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = criterion(lstm_model(xb), yb)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total_train_loss += loss.item()
    train_hist.append(total_train_loss / len(train_loader))

    # Validation phase
    lstm_model.eval(); 
    total_val_loss = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(device), yb.to(device)
            total_val_loss += criterion(lstm_model(xb), yb).item()
    val_hist.append(total_val_loss / len(val_loader))

    # Check for early stopping
    if val_hist[-1] < best_val - LSTM_MIN_DELTA:
        best_val = val_hist[-1]; patience = 0
        torch.save(lstm_model.state_dict(), MODELS_DIR / LSTM_CHECKPOINT_NAME)
    else:
        patience += 1
        if patience >= LSTM_PATIENCE:
            print(f"early stop at epoch {epoch+1}, best val {best_val:.6f}")
            break

print(f"saved {MODELS_DIR / LSTM_CHECKPOINT_NAME}")

In [ ]:
# Reload the best LSTM checkpoint and score on the same test windows
lstm_model = LSTMRulPredictor(n_features=n_features).to(device)
lstm_model.load_state_dict(
    torch.load(MODELS_DIR / LSTM_CHECKPOINT_NAME, map_location=device, weights_only=True)
)
lstm_model.eval()

preds = []
with torch.no_grad():
    for xb, _ in test_loader:
        xb = xb.to(device)
        preds.append(lstm_model(xb).cpu().numpy())
y_pred_lstm = np.concatenate(preds).ravel()

lstm_metrics = cmapss_metrics(y_test_np, y_pred_lstm)
print("LSTM test metrics:")
for k, v in lstm_metrics.items():
    print(f"  {k:>14s}: {v:.4f}")

## Comparison of results from C-MAPSS training with LSTM and SVR

The LSTM has direct access to the temporal structure of each window, while the SVR sees that same window flattened into one long vector and has to relearn cycle order from feature position. On FD001 the LSTM typically wins on every metric, with the largest gap on the asymmetric C-MAPSS score - the LSTM tends to err on the early side, which the score punishes less than late predictions.

In [ ]:
# Side-by-side metrics table for grade 5
comparison = pd.DataFrame(
    {
        "SVR":  [svr_metrics[k]  for k in ("rmse", "mae", "r2", "c_mapss_score")],
        "LSTM": [lstm_metrics[k] for k in ("rmse", "mae", "r2", "c_mapss_score")],
    },
    index=["RMSE", "MAE", "R^2", "C-MAPSS score"],
)
comparison["better"] = [
    "LSTM" if comparison.loc["RMSE",  "LSTM"] < comparison.loc["RMSE",  "SVR"] else "SVR",
    "LSTM" if comparison.loc["MAE",   "LSTM"] < comparison.loc["MAE",   "SVR"] else "SVR",
    "LSTM" if comparison.loc["R^2",   "LSTM"] > comparison.loc["R^2",   "SVR"] else "SVR",
    "LSTM" if comparison.loc["C-MAPSS score", "LSTM"] < comparison.loc["C-MAPSS score", "SVR"] else "SVR",
]
print("SVR vs LSTM on FD001 test set:")
print(comparison.round(4).to_string())

fig, ax = plt.subplots(figsize=(8, 3))

metrics_to_bar = ["RMSE", "MAE", "R^2"]
x = np.arange(len(metrics_to_bar))
w = 0.35

svr_vals  = [comparison.loc[m, "SVR"]  for m in metrics_to_bar]
lstm_vals = [comparison.loc[m, "LSTM"] for m in metrics_to_bar]

ax.bar(x - w/2, svr_vals,  w, label="SVR")
ax.bar(x + w/2, lstm_vals, w, label="LSTM")
ax.set_xticks(x); 
ax.set_xticklabels(metrics_to_bar)
ax.set_title("SVR vs LSTM - RMSE, MAE, R^2 (C-MAPSS score shown in table)")
ax.legend(); 
ax.grid(alpha=0.3, axis="y")

plt.tight_layout(); 
plt.show()